# BestShot — Chameleon Node Setup

Run this notebook from **Experiment > Jupyter Interface** on chameleoncloud.org.

Before running:
1. You must have an **active** `gpu_mi100` lease on CHI@TACC
2. Your SSH key must be uploaded to CHI@TACC

Run all cells top to bottom. At the end you will have a floating IP to SSH into.

## 1. Connect to your project

In [ ]:
from chi import server, context, lease
import os

context.version = "1.0"
context.choose_project()               # pick your CHI-XXXXXX project from the dropdown
context.choose_site(default="CHI@TACC")

## 2. Get your active lease

Change `YOUR_LEASE_NAME` to whatever you named your lease on Chameleon (e.g. `bestshot-serving`).

In [ ]:
LEASE_NAME = "serve_proj19_lease"   # change this

l = lease.get_lease(LEASE_NAME)
l.show()   # status should say ACTIVE

## 3. Launch the bare metal server

Uses the ROCm-compatible Ubuntu image (required for AMD MI100 GPU).

**Note:** if you already have a server with this name in ERROR state, delete it first in the Horizon GUI before running this cell.

In [ ]:
username = os.getenv("USER")

s = server.Server(
    f"node-bestshot-{username}",
    reservation_id=l.node_reservations[0]["id"],
    image_name="CC-Ubuntu24.04-ROCm"
)
s.submit(idempotent=True)
print("Server submitted — waiting for it to become active...")

## 4. Assign a floating IP so you can SSH in

In [ ]:
s.associate_floating_ip()

In [ ]:
s.refresh()
s.check_connectivity()

In [ ]:
s.refresh()
s.show(type="widget")   # note the floating IP in the Addresses row

## 5. Clone your repo onto the node

In [ ]:
TOKEN = "ghp_xxxx" # private repo required a token, public repo can be cloned without it

s.execute(f"git clone https://{TOKEN}@github.com/anokhimehta/bestshot.git")

## 6. Install Docker

In [ ]:
s.execute("curl -sSL https://get.docker.com/ | sudo sh")
s.execute("sudo groupadd -f docker; sudo usermod -aG docker $USER")

## 7. Verify the GPU is visible

In [ ]:
s.execute("rocm-smi")

## 8. Build your serving Docker image

In [ ]:
REPO_DIR = "bestshot/serving"    # using the folder name git clone created (bestshot), and inside where the Dockerfile is (serving)

s.execute(f"docker build -t bestshot-serve -f {REPO_DIR}/docker/Dockerfile.serve {REPO_DIR}/")

## 9. Start the serving container

In [ ]:
s.execute(
    "docker run -d --rm "
    "-p 8000:8000 "
    "--name bestshot-api "
    "bestshot-serve"
)

## 10. SSH in to run benchmarks manually

From your **local terminal**, run:

```bash
ssh -i ~/.ssh/id_rsa_chameleon cc@<FLOATING_IP>
```

Then on the node:

```bash
# CPU benchmarks
docker run --rm --network host bestshot-serve python benchmark.py --mode cpu --output results_cpu.csv

# GPU benchmarks  
docker run --rm --network host \
    --device=/dev/kfd --device=/dev/dri \
    --group-add video \
    bestshot-serve python benchmark.py --mode gpu --output results_gpu.csv

# API load test
docker run --rm --network host bestshot-serve python scripts/api_load_test.py
```

## Optional: run benchmarks directly from this notebook

Instead of SSHing in, you can also trigger commands from here using `s.execute()`.

In [ ]:
import time

# Start API server in background
s.execute(
    "docker run -d "
    "--network host "
    "--name bestshot-api "
    "bestshot-serve "
    "uvicorn app:app --host 0.0.0.0 --port 8000"
)

# Wait for server to start
time.sleep(15)

# Run benchmark against the live server
s.execute(
    "docker run --rm --network host "
    "bestshot-serve python benchmark.py"
)

# Pull results back to Jupyter environment
s.execute(f"cat benchmark_results.txt")

## 11. Delete Resources

In [ ]:
# Stop the API container if it's still running
s.execute("docker stop bestshot-api")

In [ ]:
# Delete the container 
s.execute("docker rm bestshot-api")

In [ ]:
# Shut down the bare metal server to release it back
s.delete()